**Ergodic search policy**

This policy uses an ergodic coverage controller to plan agent trajectory. Ergodic control optimizes the time-averaged spatial statistics of the agent's planned trajectory to match a target distribution, in this case the spatial distribution of predictive entropy given the current data.

More information regarding ergodic control can be found [**here**](https://github.com/MurpheyLab/ergodic-control-sandbox). 

In [ ]:
!wget -q -O box_gym.py https://raw.githubusercontent.com/MurpheyLab/boxgpt/main/box_gym.py

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation
from tqdm.auto import tqdm

from box_gym import BoxGym


def show_video(frames):
    height, width = frames[0].shape[:2]
    dpi = 100
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.axis("off")
    image = ax.imshow(frames[0])

    def update(index):
        image.set_data(frames[index])
        return (image,)

    video = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=50, blit=True
    )
    plt.close(fig)
    with plt.rc_context({"animation.embed_limit": 100.0}):
        return HTML(video.to_html5_video())

In [ ]:
class ErgodicController:
    def __init__(self, env):
        self.env = env

        num_x = env.uncertainty_grid_size
        self.dx = 1.0 / (num_x - 1)
        grids_x, grids_y = np.meshgrid(
            np.linspace(0.0, 1.0, num_x), np.linspace(0.0, 1.0, num_x)
        )
        grids = np.array([grids_x.ravel(), grids_y.ravel()]).T
        self.grids_x = grids_x
        self.grids_y = grids_y
        self.grids = grids
        self.tgt_distr = np.zeros(len(grids))

        num_k = 15
        ks_x, ks_y = np.meshgrid(np.arange(num_k), np.arange(num_k))
        ks = np.array([ks_x.ravel(), ks_y.ravel()]).T
        self.ks = ks
        self.lamks = (1.0 + np.linalg.norm(ks, axis=1)) ** -1.5
        fk_grids = np.prod(np.cos(np.pi * ks * grids[:, None]), axis=-1)
        self.hks = np.sqrt(np.sum(fk_grids**2, axis=0) * self.dx**2)
        self.fk_grids = fk_grids / self.hks
        self.cks = np.zeros(len(ks))
        self.phiks = np.zeros(len(ks))
        self.erg_dt = 0.01
        self.erg_t = 0.0

    def update_distr(self, ent_grids):
        tgt_distr = ent_grids.ravel().astype(float)
        if np.sum(tgt_distr) * self.dx**2 < 1e-12:
            tgt_distr = np.ones_like(tgt_distr)
        tgt_distr /= np.sum(tgt_distr) * self.dx**2
        self.tgt_distr = tgt_distr
        self.phiks = (
            np.sum(self.fk_grids * tgt_distr[:, None], axis=0) * self.dx**2
        )

    def plan(self, obs):
        self.update_distr(obs["uncertainty"])

        def pull2centre(x, alpha=20, c=0.02):
            c1 = c
            c2 = 1.0 - c1
            weight = np.tanh(alpha * (x - c1)) / 2
            weight += np.tanh(alpha * (c2 - x)) / 2
            dx = -np.tanh(alpha * (x - c1)) / 2
            dx += np.tanh(alpha * (c2 - x)) / 2
            return weight, dx

        sensor_pos = obs["sensor_pos"].astype(float)
        xt = sensor_pos.copy()
        ks = self.ks.copy()
        hks = self.hks.copy()
        lamks = self.lamks.copy()
        cks = self.cks.copy()
        phiks = self.phiks.copy()
        dt = self.erg_dt
        t = self.erg_t
        ud = self.env.max_velocity

        for _ in range(round(self.env.dt / self.erg_dt)):
            fk_xt = np.prod(np.cos(np.pi * ks * xt), axis=1) / hks
            cks += dt * fk_xt
            dfk_xt_all = np.array([
                -np.pi * ks[:, 0] * np.sin(np.pi * ks[:, 0] * xt[0])
                * np.cos(np.pi * ks[:, 1] * xt[1]),
                -np.pi * ks[:, 1] * np.cos(np.pi * ks[:, 0] * xt[0])
                * np.sin(np.pi * ks[:, 1] * xt[1]),
            ]) / hks
            bt = np.sum(
                lamks * (cks / (t + dt) - phiks) * dfk_xt_all, axis=1
            )
            ut = -ud * bt / (np.linalg.norm(bt) + 1e-12)

            weight_, centre_pull = pull2centre(xt)
            centre_pull *= ud / (np.linalg.norm(centre_pull) + 1e-12)
            ut = ut * weight_ + centre_pull * (1.0 - weight_)
            xt = np.clip(xt + dt * ut, 0.0, 1.0)
            t += dt

        fk_xt = np.prod(np.cos(np.pi * ks * sensor_pos), axis=1) / hks
        self.cks += self.env.dt * fk_xt
        self.erg_t += self.env.dt

        action = xt - sensor_pos
        action /= np.linalg.norm(action) + 1e-12
        action *= self.env.max_velocity
        return action.astype(np.float32)

In [ ]:
env = BoxGym()
obs, info = env.reset(seed=12)
controller = ErgodicController(env)
diagnostics = True
num_tsteps = 300
frames = [env.render(diagnostics=diagnostics)]

pbar = tqdm(range(num_tsteps))
for t in pbar:
    action = controller.plan(obs)
    obs, reward, done, truncated, info = env.step(action)
    frames.append(env.render(diagnostics=diagnostics))
    pbar.set_description(f"uncertainty: {info['uncertainty_score']:.0e}")

    if done:
        break

env.close()
print(f"Ran {info['timestep']} steps")
print(f"Final uncertainty: {info['uncertainty_score']:.0e}")
show_video(frames)